In [2]:
%reload_ext sql
%sql sqlite:///tomato.db

In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

### Orders

In [22]:
%%sql

Drop table orders;

CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    item_id INTEGER,
    quantity INTEGER NOT NULL,
    delivery_address TEXT NOT NULL,
    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO orders (user_id, item_id, quantity, delivery_address) VALUES
(1, 1, 2, '123 Main St'),
(2, 2, 1, '456 Oak Rd'),
(3, 3, 3, '789 Pine Ave');

SELECT * FROM orders;

 * sqlite:///tomato.db
Done.
Done.
3 rows affected.
Done.


order_id,user_id,item_id,quantity,delivery_address,order_date
1,1,1,2,123 Main St,2026-03-26 14:51:33
2,2,2,1,456 Oak Rd,2026-03-26 14:51:33
3,3,3,3,789 Pine Ave,2026-03-26 14:51:33


In [4]:
%%sql

Select * from orders;

 * sqlite:///tomato.db
Done.


order_id,user_id,item_id,quantity,delivery_address,order_date
1,1,1,2,123 Main St,2026-03-26 14:51:33
2,2,2,1,456 Oak Rd,2026-03-26 14:51:33
3,3,3,3,789 Pine Ave,2026-03-26 14:51:33


### Categories

In [12]:
%%sql

Drop table categories;

CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name VARCHAR(255) NOT NULL
);

INSERT INTO categories (category_name) VALUES
('Vegetarian'),
('Non-Vegetarian'),
('Vegan');

Select * from categories;

 * sqlite:///tomato.db
Done.
Done.
3 rows affected.
Done.


category_id,category_name
1,Vegetarian
2,Non-Vegetarian
3,Vegan


### Items

In [13]:
%%sql

drop table items;

CREATE TABLE IF NOT EXISTS items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_name VARCHAR(255) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    category_id INT
);

INSERT INTO items (item_name, price, category_id) VALUES
('Biriyani', 150.00, 2),  
('Paneer', 100.00, 1),  
('Butter Chicken', 200.00, 2); 

Select * from items;

 * sqlite:///tomato.db
Done.
Done.
3 rows affected.
Done.


item_id,item_name,price,category_id
1,Biriyani,150,2
2,Paneer,100,1
3,Butter Chicken,200,2


### Order Items

In [11]:
%%sql

Drop table order_items;

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INT,
    item_id INT,
    quantity INT NOT NULL,
    price DECIMAL(10, 2),
    FOREIGN KEY (item_id) REFERENCES items(item_id)
);

INSERT INTO order_items (order_id, item_id, quantity, price) VALUES
(1, 1, 2, 150.00),
(2, 2, 1, 100.00),
(3, 3, 3, 200.00);

Select * from order_items;

 * sqlite:///tomato.db
Done.
Done.
3 rows affected.
Done.


order_item_id,order_id,item_id,quantity,price
1,1,1,2,150
2,2,2,1,100
3,3,3,3,200


### Payments

In [15]:
%%sql

Drop table payments;

CREATE TABLE IF NOT EXISTS payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INT,
    payment_method VARCHAR(50),
    payment_status VARCHAR(50),
    amount DECIMAL(10, 2),
    paid_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO payments (order_id, payment_method, payment_status, amount) VALUES
(1, 'Credit Card', 'Paid', 300.00),
(2, 'Debit Card', 'Paid', 100.00),
(3, 'GPay', 'Pending', 600.00);

Select * from payments;

 * sqlite:///tomato.db
Done.
Done.
3 rows affected.
Done.


payment_id,order_id,payment_method,payment_status,amount,paid_at
1,1,Credit Card,Paid,300,2026-03-27 06:16:18
2,2,Debit Card,Paid,100,2026-03-27 06:16:18
3,3,GPay,Pending,600,2026-03-27 06:16:18


### Customers

In [30]:
%%sql

CREATE TABLE IF NOT EXISTS customers (
    user_id INT PRIMARY KEY,
    first_name VARCHAR(255),
    last_name VARCHAR(255),
    phone_number VARCHAR(15),
    email VARCHAR(255)
);

INSERT INTO customers (user_id, first_name, last_name, phone_number, email) VALUES
(1, 'John', 'Doe', '123-456-7890', 'john.doe@example.com'),
(2, 'Jane', 'Smith', '987-654-3210', 'jane.smith@example.com'),
(3, 'Alice', 'Johnson', '111-222-3333', 'alice.johnson@example.com'),
(4, 'Bob', 'Brown', '444-555-6666', 'bob.brown@example.com'),
(5, 'Gowtham', 'Kumar', '777-888-9999', 'gowtham.kumar@example.com');

select * from customers;

 * sqlite:///tomato.db
Done.
5 rows affected.
Done.


user_id,first_name,last_name,phone_number,email
1,John,Doe,123-456-7890,john.doe@example.com
2,Jane,Smith,987-654-3210,jane.smith@example.com
3,Alice,Johnson,111-222-3333,alice.johnson@example.com
4,Bob,Brown,444-555-6666,bob.brown@example.com
5,Gowtham,Kumar,777-888-9999,gowtham.kumar@example.com


## ETL
### 1. Total Revenue from all orders.

In [16]:
%%sql

SELECT SUM(price * quantity) AS total_revenue
FROM order_items;

 * sqlite:///tomato.db
Done.


total_revenue
1000


### 2. Revenue by Item

In [17]:
%%sql

SELECT i.item_name, SUM(oi.price * oi.quantity) AS total_revenue
FROM order_items oi
JOIN items i 
ON oi.item_id = i.item_id
GROUP BY i.item_name
ORDER BY total_revenue DESC;

 * sqlite:///tomato.db
Done.


item_name,total_revenue
Butter Chicken,600
Biriyani,300
Paneer,100


### 3. Revenue by Payment Method

In [18]:
%%sql

SELECT p.payment_method, SUM(p.amount) AS total_revenue
FROM payments p
JOIN orders o ON p.order_id = o.order_id
GROUP BY p.payment_method;

 * sqlite:///tomato.db
Done.


payment_method,total_revenue
Credit Card,300
Debit Card,100
GPay,600


### 4. Total Revenue by Date

In [19]:
%%sql

SELECT DATE(p.paid_at) AS date, SUM(p.amount) AS daily_revenue
FROM payments p
GROUP BY DATE(p.paid_at)
ORDER BY date;

 * sqlite:///tomato.db
Done.


date,daily_revenue
2026-03-27,1000


### 5. Total Orders and Revenue by User

In [20]:
%%sql

SELECT 
    u.first_name,
    u.last_name,
    u.phone_number,
    u.email,
    COUNT(o.order_id) AS total_orders,
    SUM(oi.price * oi.quantity) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers u ON o.user_id = u.user_id
GROUP BY u.user_id
ORDER BY total_revenue DESC;

 * sqlite:///tomato.db
Done.


first_name,last_name,phone_number,email,total_orders,total_revenue
Alice,Johnson,111-222-3333,alice.johnson@example.com,1,600
John,Doe,123-456-7890,john.doe@example.com,1,300
Jane,Smith,987-654-3210,jane.smith@example.com,1,100


### 6. Items Ordered by Category

In [21]:
%%sql

SELECT c.category_name, i.item_name, SUM(oi.quantity) AS total_quantity_ordered
FROM order_items oi
JOIN items i ON oi.item_id = i.item_id
JOIN categories c ON i.category_id = c.category_id
GROUP BY c.category_name, i.item_name
ORDER BY total_quantity_ordered DESC;

 * sqlite:///tomato.db
Done.


category_name,item_name,total_quantity_ordered
Non-Vegetarian,Butter Chicken,3
Non-Vegetarian,Biriyani,2
Vegetarian,Paneer,1


### 7. Orders by Payment Status

In [22]:
%%sql

SELECT p.payment_status, COUNT(o.order_id) AS total_orders
FROM payments p
JOIN orders o ON p.order_id = o.order_id
GROUP BY p.payment_status;

 * sqlite:///tomato.db
Done.


payment_status,total_orders
Paid,2
Pending,1


### 8. Users with Most Orders

In [24]:
%%sql

SELECT 
    u.first_name, 
    u.last_name, 
    COUNT(o.order_id) AS total_orders
FROM customers u
JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id
ORDER BY total_orders DESC
LIMIT 5;

 * sqlite:///tomato.db
Done.


first_name,last_name,total_orders
Alice,Johnson,1
Jane,Smith,1
John,Doe,1


### 9. Revenue by Category

In [30]:
%%sql

SELECT c.category_name, SUM(oi.price * oi.quantity) AS total_revenue
FROM order_items oi
JOIN items i ON oi.item_id = i.item_id
JOIN categories c ON i.category_id
GROUP BY c.category_name;

 * sqlite:///tomato.db
Done.


category_name,total_revenue
Non-Vegetarian,1000
Vegan,1000
Vegetarian,1000


### 10. Items Purchased in Specific Order

In [27]:
%%sql

SELECT oi.order_id, i.item_name, oi.quantity, oi.price
FROM order_items oi
JOIN items i ON oi.item_id = i.item_id
WHERE oi.order_id = 1;

 * sqlite:///tomato.db
Done.


order_id,item_name,quantity,price
1,Biriyani,2,150


### 11. Customer Details with Orders

In [28]:
%%sql

SELECT 
    c.first_name, 
    c.last_name, 
    c.phone_number, 
    c.email, 
    o.order_id, 
    o.delivery_address, 
    oi.item_id, 
    oi.quantity
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.user_id = c.user_id;

 * sqlite:///tomato.db
Done.


first_name,last_name,phone_number,email,order_id,delivery_address,item_id,quantity
John,Doe,123-456-7890,john.doe@example.com,1,123 Main St,1,2
Jane,Smith,987-654-3210,jane.smith@example.com,2,456 Oak Rd,2,1
Alice,Johnson,111-222-3333,alice.johnson@example.com,3,789 Pine Ave,3,3


### 12. Revenue by Customer

In [29]:
%%sql

SELECT 
    c.first_name, 
    c.last_name, 
    c.phone_number, 
    c.email, 
    SUM(oi.price * oi.quantity) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.user_id = c.user_id
GROUP BY c.user_id
ORDER BY total_revenue DESC;

 * sqlite:///tomato.db
Done.


first_name,last_name,phone_number,email,total_revenue
Alice,Johnson,111-222-3333,alice.johnson@example.com,600
John,Doe,123-456-7890,john.doe@example.com,300
Jane,Smith,987-654-3210,jane.smith@example.com,100
